In [ ]:
%pip install -q -U pip
%pip install -q fastf1 scikit-learn

In [ ]:
from pathlib import Path

cache_dir = Path("f1_cache")

# Create the directory if it doesn’t exist
cache_dir.mkdir(exist_ok=True)

print(f"Directory '{cache_dir}' is ready to use.")

In [ ]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [ ]:
# Enable FastF1 caching
fastf1.Cache.enable_cache("f1_cache")

# Load 2025 Australian GP Practice and Qualifying session
session_2025_FP1 = fastf1.get_session(2025, "Australia", "FP1")
session_2025_FP1.load()

session_2025_FP2 = fastf1.get_session(2025, "Australia", "FP2")
session_2025_FP2.load()

session_2025_FP3 = fastf1.get_session(2025, "Australia", "FP3")
session_2025_FP3.load()

session_2025_Q = fastf1.get_session(2025, "Australia", "Q")
session_2025_Q.load()


In [ ]:
session_2025_FP1.laps['SessionType'] = "FP1"
session_2025_FP2.laps['SessionType'] = "FP2"
session_2025_FP3.laps['SessionType'] = "FP3"
session_2025_Q.laps['SessionType'] = "Q"

In [ ]:
# Drop all rows with Deleted lap times.
def drop_deleted_laps(session):
    laps = session[session['Deleted'] == False]

    return laps

In [ ]:
session_2025_FP1 = drop_deleted_laps(session_2025_FP1.laps)
session_2025_FP2 = drop_deleted_laps(session_2025_FP2.laps)
session_2025_FP3 = drop_deleted_laps(session_2025_FP3.laps)
session_2025_Q = drop_deleted_laps(session_2025_Q.laps)

In [ ]:
def lap_time_process(session):
    # Extract lap and sector times
    laps = session[["Driver", "DriverNumber", "LapTime", "SessionType"]].copy()
    laps.dropna(inplace=True)

    # Convert times to seconds
    for col in ["LapTime"]:
        laps[f"{col} (s)"] = laps[col].dt.total_seconds()

    # Group by driver to get average sector times per driver
    sector_times = laps.groupby(["DriverNumber", "SessionType"])[["LapTime (s)"]].min().reset_index()

    return sector_times

In [ ]:
session_2025_FP1_clean = lap_time_process(session_2025_FP1)
session_2025_FP2_clean = lap_time_process(session_2025_FP2)
session_2025_FP3_clean = lap_time_process(session_2025_FP3)
session_2025_Q_clean = lap_time_process(session_2025_Q)

In [ ]:
# Compile all Practice sessions lap times
FP_lap_times = pd.concat([
    session_2025_FP1_clean,
    session_2025_FP2_clean,
    session_2025_FP3_clean,
    session_2025_Q_clean
], ignore_index=True)

# Pivot table to make FP1, FP2, FP3 and Q columns
session_2025_all = FP_lap_times.pivot(
    index='DriverNumber',
    columns='SessionType',
    values='LapTime (s)'
)

session_2025_all.columns.name = None

# Fill up NaN values with the mean of other sessions.
session_2025_all.iloc[:, 0:] = session_2025_all.iloc[:, 0:].apply(
    lambda row: row.fillna(row.mean()), 
    axis=1
)

session_2025_all = session_2025_all.reset_index()

session_2025_all

In [ ]:
# Feature selection and target variable selection.
X = session_2025_all[['DriverNumber', 'FP1', 'FP2', 'FP3']]
y = session_2025_all['Q']

In [ ]:
# Train Gradient Boosting Model

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=38)
model = GradientBoostingRegressor(n_estimators=10, learning_rate=0.1, max_depth=2, random_state=38)
model.fit(X_train, y_train)

In [ ]:
# Testing the model with unseen data
y_pred = model.predict(X_test)

# Performance Metrics calculation
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print results
print("Model Performance Metrics:")
print(f"R² Score: {r2:.2f}")
print(f"RMSE: {rmse:.2f} seconds")
print(f"MAE: {mae:.2f} seconds")
print(f"MSE: {mse:.2f}")

# Using model to predict 2026 Australian GP Qualifying results.

In [ ]:
# Practice session Data for 2026 Australian GP
practice_2026 = pd.DataFrame({
    "DriverNumber": [81, 63, 4, 1, 
                    44, 16, 6, 12, 
                    22, 23, 31, 27, 
                    14, 18, 55, 10, 
                    87, 43, 5, 30],

    "FP1":  [77.180, 77.481, 77.096, 77.481, 
            77.973, 77.755, 78.175, 77.235, 
            78.670, 78.737, 78.147, 78.579, 
            76.453, 76.483, 76.062, 77.980, 
            79.785, 79.123, 79.520, 79.094],

    "FP2":  [76.180, 76.481, 76.096, 76.481, 
            76.973, 76.755, 77.175, 76.235, 
            77.670, 77.737, 77.147, 77.579, 
            75.453, 75.483, 75.062, 76.980, 
            78.785, 78.123, 78.520, 78.094],

    "FP3":  [75.180, 75.481, 75.096, 75.481, 
            75.973, 75.755, 76.175, 75.235, 
            76.670, 76.737, 76.147, 76.579, 
            74.453, 74.483, 74.062, 75.980, 
            77.785, 77.123, 77.520, 77.094],
})

# Map Driver numbers to Driver name
driver_mapping = {
    81: "Oscar Piastri", 63: "George Russell", 4: "Lando Norris", 1: "Max Verstappen",
    44: "Lewis Hamilton", 16: "Charles Leclerc", 6: "Isack Hadjar", 12: "Andrea Kimi Antonelli",
    22: "Yuki Tsunoda", 23: "Alexander Albon", 31: "Esteban Ocon", 27: "Nico Hülkenberg",
    14: "Fernando Alonso", 18: "Lance Stroll", 55: "Carlos Sainz Jr.", 10: "Pierre Gasly",
    87: "Oliver Bearman", 43: "Franco Colapinto", 5: "Gabriel Bortoleto", 30: "Liam Lawson"
}

# Print input data
practice_2026

In [ ]:
# Predict qualifying result using practice lap times
print("\n🏁 Predicted 2026 Australian GP Qualifying results🏁\n")
predicted_qualifying_times = model.predict(practice_2026)

qualifying_2026 = pd.DataFrame()
qualifying_2026['Driver Number'] = practice_2026['DriverNumber']
qualifying_2026["Predicted Qualifying Time (s)"] = predicted_qualifying_times

# Map driver numbers to names
qualifying_2026['Driver Name'] = practice_2026['DriverNumber'].map(driver_mapping)

# Rank drivers by predicted race time
qualifying_2026 = qualifying_2026.sort_values(by="Predicted Qualifying Time (s)").reset_index(drop=True)

# Add grid position
qualifying_2026['Grid Position'] = range(1, len(qualifying_2026) + 1)

# Reorder columns to show name first
result = qualifying_2026[['Grid Position','Driver Name', 'Driver Number', 'Predicted Qualifying Time (s)']]

# # Display results
result